<h1><center>Laboratorio 9: Optimización de modelos 💯</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2024</strong></center>

### **Cuerpo Docente:**

- Profesores: Ignacio Meza, Sebastián Tinoco
- Auxiliar: Eduardo Moya
- Ayudantes: Nicolás Ojeda, Melanie Peña, Valentina Rojas

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Damián De Aguiar
- Nombre de alumno 2: Angelo León


### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/damiatus/labPrograCientifica)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.

### Reglas:

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibidas las copias.
- Pueden usar cualquer matrial del curso que estimen conveniente.
- Código que no se pueda ejecutar, no será revisado.

### Objetivos principales del laboratorio

- Optimizar modelos usando `optuna`
- Recurrir a técnicas de *prunning*
- Forzar el aprendizaje de relaciones entre variables mediante *constraints*
- Fijar un pipeline con un modelo base que luego se irá optimizando.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

In [1]:
#!pip install -qq xgboost optuna

print('jola')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.8/362.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.2/233.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.5 MB/s eta 0:00:00
jola


# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [35]:
import pandas as pd
import numpy as np
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
file_path = '/content/drive/My Drive/LabdeProgra/Lab9'
df = pd.read_csv(file_path + "/sales.csv")
df.head()

,id,date,city,lat,long,pop,shop,brand,container,capacity,price,quantity
0,0,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,glass,500ml,0.96,13280
1,1,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,plastic,1.5lt,2.86,6727
2,2,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,can,330ml,0.87,9848
3,3,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,glass,500ml,1.00,20050
4,4,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,can,330ml,0.39,25696


## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [49]:
#1:
from sklearn import set_config

set_config(transform_output="pandas")

train_data, temp_data = train_test_split(df, test_size=0.3, random_state=20676305)
val_data, test_data = train_test_split(temp_data, test_size=0.33, random_state=20676305)

X_train = train_data.drop(columns=['quantity'])
y_train = train_data['quantity']

X_val = val_data.drop(columns=['quantity'])
y_val = val_data['quantity']

X_test = test_data.drop(columns=['quantity'])
y_test = test_data['quantity']

print(f"Train size: {train_data.shape}, Validation size: {val_data.shape}, Test size: {test_data.shape}")

Train size: (5219, 12), Validation size: (1498, 12), Test size: (739, 12)


In [50]:
# 2:

def extract_date_features(df):
    df['year'] = pd.to_datetime(df['date'], format='%d/%m/%y').dt.year
    df['month'] = pd.to_datetime(df['date'], format='%d/%m/%y').dt.month
    df['day'] = pd.to_datetime(df['date'], format='%d/%m/%y').dt.day

    df['year'] = df['year'].astype('category')
    df['month'] = df['month'].astype('category')
    df['day'] = df['day'].astype('category')
    return df.drop(columns=['date'])


date_transformer = FunctionTransformer(extract_date_features)

In [25]:
df.nunique()

,0
id,7456
date,84
city,5
lat,6
long,6
pop,35
shop,6
brand,5
container,3
capacity,3


Al analizar los valores únicos vemos que las variables que no son categoricas son: 'lat', 'long', 'pop', 'price', 'quantity'

In [57]:
df

,id,date,city,lat,long,pop,shop,brand,container,capacity,price,quantity
0,0,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,glass,500ml,0.96,13280
1,1,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,plastic,1.5lt,2.86,6727
2,2,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,can,330ml,0.87,9848
3,3,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,glass,500ml,1.00,20050
4,4,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,can,330ml,0.39,25696
...,...,...,...,...,...,...,...,...,...,...,...,...
7451,7555,31/12/18,Athens,37.97945,23.71622,664046,shop_1,kinder-cola,plastic,1.5lt,2.52,13760
7452,7556,31/12/18,Athens,37.97945,23.71622,664046,shop_1,orange-power,plastic,1.5lt,2.18,16309
7453,7557,31/12/18,Patra,38.24444,21.73444,168034,shop_6,kinder-cola,can,330ml,0.85,24378
7454,7558,31/12/18,Thessaloniki,40.64361,22.93086,354290,shop_4,adult-cola,plastic,1.5lt,2.17,20691


In [51]:
# 3:

num_features = ['lat', 'long', 'pop', 'price']
cat_features = ['city', 'shop', 'year', 'month', 'day']

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ]
)

In [52]:
# 4 y 5:

pipeline_dummy = Pipeline(steps=[
    ('date_extractor', date_transformer),
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy="mean"))
])

pipeline_dummy.fit(X_train, y_train)
y_val_pred = pipeline_dummy.predict(X_val)

mae_dummy = mean_absolute_error(y_val, y_val_pred)
print(f'MAE en validación con DummyRegressor: {mae_dummy}')

MAE en validación con DummyRegressor: 13270.306990274572


El MAE reportado es e 13.270,30; eso quiere decir que las prediciones del modelo utilizado poseen un error medio absoluto de tal valor. Como la unidad que se esta prediciendo es en cantidades, el error tambien posee dicha unidad, que es una de las caracteristicas del MAE.

Crear el Pipeline con DummyRegressor

In [53]:
# 6:

pipeline_xgb = Pipeline(steps=[
    ('date_extractor', date_transformer),
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor())
])

pipeline_xgb.fit(X_train, y_train)
y_val_pred_xgb = pipeline_xgb.predict(X_val)

mae_xgb = mean_absolute_error(y_val, y_val_pred_xgb)
print(f'MAE en validación con XGBRegressor: {mae_xgb}')

MAE en validación con XGBRegressor: 7002.643389428092


Dado que el MAE es el error medio que tiene el model0 entrenado, mientras menor sea, es un indicio de que el modelo es mejor. Dicho esto, como el MAE de XGBRegressor es menor a DummyRegressor, el modelo es mejor.

In [56]:
# 7:
import pickle

with open(file_path + '/pipeline_dummy.pkl', 'wb') as f:
    pickle.dump(pipeline_dummy, f)

with open(file_path + '/pipeline_xgb.pkl', 'wb') as f:
    pickle.dump(pipeline_xgb, f)

## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

In [64]:
#1:

preprocessor.fit(X_train)
feature_names = preprocessor.get_feature_names_out()

monotone_constraints = {name: 0 for name in feature_names}
monotone_constraints["num__price"] = -1

pipeline_xgb_monotonic = Pipeline(steps=[
    ('date_extractor', date_transformer),
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(monotone_constraints=monotone_constraints))
])

pipeline_xgb_monotonic.fit(X_train, y_train)

Pipeline(steps=[('date_extractor',
                 FunctionTransformer(func=<function extract_date_features at 0x7fe92cfc7010>)),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['lat', 'long', 'pop',
                                                   'price']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['city', 'shop', 'year',
                                                   'month', 'day'])])),
                ('regressor',
                 XGBRegressor(base_s...
                                                    'cat__month_6': 0,
                                                    'cat__month_7': 0,
                                                    'cat__month_8': 0,
                                                    'cat__month_9': 0,
                                                    'cat__shop_shop_1': 0,
                                                    'cat__shop_shop_2': 0,
                                                    'cat__shop_shop_3': 0,
                                                    'cat__shop_shop_4': 0,
                                                    'cat__shop_shop_5': 0,
                                                    'cat__shop_shop_6': 0,
                                                    'cat__year_2012': 0,
                                                    'cat__year_2013': 0,
                                                    'cat__year_2014': 0, ...},
                              multi_strategy=None, n_estimators=None,
                              n_jobs=None, num_parallel_tree=None,
                              random_state=None, ...))])

In [65]:
#2:
y_val_pred_monotonic = pipeline_xgb_monotonic.predict(X_val)
mae_monotonic = mean_absolute_error(y_val, y_val_pred_monotonic)

print(f'MAE en validación con XGBRegressor (monotonía negativa): {mae_monotonic}')

MAE en validación con XGBRegressor (monotonía negativa): 7289.645706375387


In [69]:
#3:

print(f'MAE anterior: {mae_xgb}')
print(f'MAE con XGBRegressor y restricción: {mae_monotonic}')

MAE anterior: 7002.643389428092
MAE con XGBRegressor y restricción: 7289.645706375387


Dado los resultados, se puede decir que mi amigo no tenia razon, ya que el nuevo MAE es mayor al anterior, lo que quiere decir que el modelo posee un error medio mayor que el anterior modelo.

In [70]:
#4:

with open(file_path + '/pipeline_xgb_monotonic.pkl', 'wb') as f:
    pickle.dump(pipeline_xgb_monotonic, f)

## 1.3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [135]:
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
    n_estimators = trial.suggest_int("n_estimators", 50, 1000)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    max_leaves = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 0, 1)
    reg_lambda = trial.suggest_float("reg_lambda", 0, 1)
    min_frequency = trial.suggest_float("min_frequency", 0.0, 1.0)

    one_hot_encoder = OneHotEncoder(min_frequency=min_frequency, sparse_output=False)

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', num_features),
            ('cat', one_hot_encoder, cat_features)
        ],
        remainder='drop'
    )

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', xgb.XGBRegressor(
            learning_rate=learning_rate,
            n_estimators=n_estimators,
            max_depth=max_depth,
            max_leaves=max_leaves,
            min_child_weight=min_child_weight,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=20676305
        ))
    ])

    pipeline.fit(X_train, y_train)

    y_val_pred = pipeline.predict(X_val)
    mae = mean_absolute_error(y_val, y_val_pred)
    if "best_mae" not in study.user_attrs or mae < study.user_attrs["best_mae"]:
        study.set_user_attr("best_mae", mae)
        study.set_user_attr("best_pipeline", pipeline)

    return mae

In [136]:
#2:

study = optuna.create_study(sampler=TPESampler(), direction='minimize')
study.optimize(objective, timeout=300)

In [137]:
#3:

print(f'Número de trials: {len(study.trials)}')
print(f'MAE: {study.best_value}')
print(f'Mejores hiperparámetros: {study.best_params}')

Número de trials: 372
MAE: 6455.685521613454
Mejores hiperparámetros: {'learning_rate': 0.06088478489392484, 'n_estimators': 542, 'max_depth': 4, 'max_leaves': 6, 'min_child_weight': 3, 'reg_alpha': 0.7741164547571104, 'reg_lambda': 0.20572736310470496, 'min_frequency': 0.02018803593966867}


Comparando los modelos anteriores con este nuevo que fue optimizado mediante Optuna sus hiperparametros, se puede decir que los resultados, medidos en el MAE, son mejores que los anteriores modelos. Esta mejora se debe a la optimizacion producida, al probar los distintos hiperparametros, esto provoca que los modelos posean mejores resultados.

4: Hiperparametros.

Hiperparámetros de XGBRegressor

- learning_rate: Parámetro controla la contribución de cada árbol al modelo final. Un valor más bajo significa que el modelo necesita más árboles para lograr la misma precisión, pero puede llevar a un modelo más robusto y menos propenso a sobreajustarse.

- n_estimators: Define el número total de árboles que se construirán en el modelo. Un número mayor puede aumentar el rendimiento, pero también puede llevar a un sobreajuste.

- max_depth: Determina la profundidad máxima de cada árbol. Árboles más profundos pueden capturar patrones complejos en los datos, pero también son más propensos a sobreajustarse. Se ajusta este parámetro para encontrar un balance entre sesgo y varianza.

- max_leaves: Establece el número máximo de hojas que se pueden crear en cada árbol. Limitar las hojas puede ayudar a reducir la complejidad del modelo y evitar el sobreajuste.

- min_child_weight: Define el peso mínimo de la suma de instancias en un nodo hijo. Valores más altos previenen la creación de nodos que contienen pocos ejemplos, ayudando a controlar el sobreajuste.

- reg_alpha: Regularización L1. Penaliza la complejidad del modelo. Un valor mayor puede ayudar a reducir el sobreajuste al obligar al modelo a aprender representaciones más simples.

- reg_lambda: Regularización L2. Similar a reg_alpha, pero la regularización L2 penaliza más fuertemente los pesos grandes en el modelo, lo que también puede ayudar a prevenir el sobreajuste.

Hiperparámetro de OneHotEncoder:

- min_frequency: Define la frecuencia mínima requerida para que una categoría se incluya en el conjunto de datos transformado. Si una categoría aparece menos veces que el valor especificado, se agrupa en una categoría "poco frecuente". Puede ser útil para reducir la dimensionalidad y prevenir el sobreajuste.

In [138]:
#5:

best_pipeline = study.user_attrs["best_pipeline"]
with open(file_path + '/best_xgb_pipeline.pkl', 'wb') as f:
    pickle.dump(best_pipeline, f)

## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

In [89]:
#!pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 2.0 MB/s eta 0:00:00


1: El prunning es una tecnica usada en la optimizacion de hiperparametros, con el enfoque de parar el entrenamiento del modelo si no hay mejorias en sus resultados. Esta tecnica posee como objetivo ahorrar recursos computacionales y reducir los tiempos de optimizacion de hiperparametros.

In [130]:
#2:

from optuna.integration import XGBoostPruningCallback
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective2(trial):
    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
    n_estimators = trial.suggest_int("n_estimators", 50, 1000)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    max_leaves = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 0, 1)
    reg_lambda = trial.suggest_float("reg_lambda", 0, 1)
    min_frequency = trial.suggest_float("min_frequency", 0.0, 1.0)

    one_hot_encoder = OneHotEncoder(min_frequency=min_frequency, sparse_output=False)

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', num_features),
            ('cat', one_hot_encoder, cat_features)
        ],
        remainder='drop'
    )

    X_train_transformed = preprocessor.fit_transform(X_train)
    X_val_transformed = preprocessor.transform(X_val)

    dtrain = xgb.DMatrix(X_train_transformed, label=y_train)
    dval = xgb.DMatrix(X_val_transformed, label=y_val)

    params = {
        'learning_rate': learning_rate,
        'max_depth': max_depth,
        'max_leaves': max_leaves,
        'min_child_weight': min_child_weight,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda,
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'random_state': 20676305
    }

    model = xgb.train(
        params,
        dtrain,
        evals=[(dval, 'validation')],
        callbacks=[XGBoostPruningCallback(trial, observation_key='validation-mae')]
    )

    y_val_pred = model.predict(dval)
    mae = mean_absolute_error(y_val, y_val_pred)
    trial.report(mae, step=1)

    if trial.should_prune():
        raise optuna.TrialPruned()

    if "best_mae" not in study.user_attrs or mae < study.user_attrs["best_mae"]:
        study.set_user_attr("best_mae", mae)
        study.set_user_attr("best_pipeline", model)

    return mae


In [131]:
#3:

study = optuna.create_study(sampler=optuna.samplers.TPESampler(), direction='minimize')
study.optimize(objective2, timeout=300, show_progress_bar=True)

   0%|          | 00:00/05:00

[0]	validation-mae:13165.97187
[1]	validation-mae:13063.99593
[2]	validation-mae:12964.68292
[3]	validation-mae:12867.17973
[4]	validation-mae:12771.93425
[5]	validation-mae:12679.67755
[6]	validation-mae:12589.49453
[7]	validation-mae:12501.16954
[8]	validation-mae:12415.75645
[9]	validation-mae:12332.05000


/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 1 is already reported.
  warnings.warn(


[0]	validation-mae:13242.56970
[1]	validation-mae:13214.93281
[2]	validation-mae:13187.40374
[3]	validation-mae:13160.00397
[4]	validation-mae:13132.72814
[5]	validation-mae:13105.71719
[6]	validation-mae:13079.18700
[7]	validation-mae:13052.93078
[8]	validation-mae:13026.76051
[9]	validation-mae:13000.93780
[0]	validation-mae:13252.34074
[1]	validation-mae:13234.43055
[2]	validation-mae:13216.57751
[3]	validation-mae:13198.84799
[4]	validation-mae:13181.10459
[5]	validation-mae:13163.50939
[6]	validation-mae:13145.91212


/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 1 is already reported.
  warnings.warn(


...

[0]	validation-mae:12405.63724
[1]	validation-mae:11688.20780
[2]	validation-mae:11053.23722
[3]	validation-mae:10520.56826
[4]	validation-mae:10038.44388
[5]	validation-mae:9622.36401
[6]	validation-mae:9258.66362
[7]	validation-mae:8960.36240
[8]	validation-mae:8686.26386
[9]	validation-mae:8427.50340


/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 1 is already reported.
  warnings.warn(


[0]	validation-mae:12429.41353
[0]	validation-mae:12447.67192
[0]	validation-mae:12430.15195
[0]	validation-mae:12455.62800
[0]	validation-mae:12416.21153
[1]	validation-mae:11696.71634
[2]	validation-mae:11056.65105
[3]	validation-mae:10516.73839
[4]	validation-mae:10035.90280
[5]	validation-mae:9616.02728
[6]	validation-mae:9239.39218
[7]	validation-mae:8936.73336
[8]	validation-mae:8668.73374
[9]	validation-mae:8412.86607


/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 1 is already reported.
  warnings.warn(


[0]	validation-mae:13056.95170
[0]	validation-mae:12472.24181
[0]	validation-mae:12416.37252
[1]	validation-mae:11692.77681
[2]	validation-mae:11069.59227
[3]	validation-mae:10523.41480
[0]	validation-mae:12443.56981


In [132]:
#4:

print("Número de trials:", len(study.trials))
print("MEJOR MAE:", study.best_value)
print("Mejores hiperparámetros:", study.best_params)

Número de trials: 867
MEJOR MAE: 8295.530641767147
Mejores hiperparámetros: {'learning_rate': 0.09997253101991131, 'n_estimators': 411, 'max_depth': 8, 'max_leaves': 0, 'min_child_weight': 4, 'reg_alpha': 0.8212350405694226, 'reg_lambda': 0.385110921857438, 'min_frequency': 0.03395392955562982}


Al comparar con la anterior version de Optuna, sin prunning, se puede ver que los resultados no son mejores. Esto se debe a que mediante el prunning, se decide detener el entrenamiento del modelo si no hay mejorias en las metricas para evaluar el modelo, en nuestro caso, el MAE.

In [134]:
#5:

best_pipeline2 = study.user_attrs["best_pipeline"]
with open(file_path + '/best_xgb_pipeline2.pkl', 'wb') as f:
    pickle.dump(best_pipeline2, f)

## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [139]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision


import optuna

# You can use Matplotlib instead of Plotly for visualization by simply replacing `optuna.visualization` with
# `optuna.visualization.matplotlib` in the following examples.
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank
from optuna.visualization import plot_slice
from optuna.visualization import plot_timeline


SEED = 13
torch.manual_seed(SEED)

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DIR = ".."
BATCHSIZE = 128
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10


def define_model(trial):
    n_layers = trial.suggest_int("n_layers", 1, 2)
    layers = []

    in_features = 28 * 28
    for i in range(n_layers):
        out_features = trial.suggest_int("n_units_l{}".format(i), 64, 512)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.ReLU())

        in_features = out_features

    layers.append(nn.Linear(in_features, 10))
    layers.append(nn.LogSoftmax(dim=1))

    return nn.Sequential(*layers)


# Defines training and evaluation.
def train_model(model, optimizer, train_loader):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.view(-1, 28 * 28).to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        F.nll_loss(model(data), target).backward()
        optimizer.step()


def eval_model(model, valid_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(valid_loader):
            data, target = data.view(-1, 28 * 28).to(DEVICE), target.to(DEVICE)
            pred = model(data).argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    accuracy = correct / N_VALID_EXAMPLES

    return accuracy

In [140]:
def objective(trial):
    train_dataset = torchvision.datasets.FashionMNIST(
        DIR, train=True, download=True, transform=torchvision.transforms.ToTensor()
    )
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(train_dataset, list(range(N_TRAIN_EXAMPLES))),
        batch_size=BATCHSIZE,
        shuffle=True,
    )

    val_dataset = torchvision.datasets.FashionMNIST(
        DIR, train=False, transform=torchvision.transforms.ToTensor()
    )
    val_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(val_dataset, list(range(N_VALID_EXAMPLES))),
        batch_size=BATCHSIZE,
        shuffle=True,
    )
    model = define_model(trial).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(), trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    )

    for epoch in range(10):
        train_model(model, optimizer, train_loader)

        val_accuracy = eval_model(model, val_loader)
        trial.report(val_accuracy, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return val_accuracy

In [141]:
plot_optimization_history(study)

Se ve que a partir del trial 11 el valor baja, por lo que los primeros trial son mejores.

In [142]:
plot_parallel_coordinate(study)

In [144]:
plot_param_importances(study)

## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por mail o U-cursos.

<p align="center">
  <img src="https://media.tenor.com/8CT1AXElF_cAAAAC/gojo-satoru.gif">
</p>

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=87110296-876e-426f-b91d-aaf681223468' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>